### vLLM for efficient serving

In [15]:
from vllm import LLM, SamplingParams

In [16]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [17]:
#serve vllm thorugh openai api servers & make concurent calls

In [18]:
#!vllm serve lora/lora_16bit_merged_3b_r128_s1000_i1000_v1 --max-model-len=1024 --dtype auto --api-key my-api-key

### OpenAI style client

In [32]:
from openai import OpenAI
# Set OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "my-api-key"
openai_api_base = "http://localhost:8000/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)



def get_reponse(description):
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. 
        Write a response that appropriately completes the request.

                ### Instruction:
                Please write a SVG code for the given input.

                ### Input:
                {}

                ### Response:
                """
    
    formatted_input = alpaca_prompt.format(description)
    chat_response = client.chat.completions.create(
        model="lora/lora_16bit_merged_3b_r128_s1000_i1000_v1",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"{formatted_input}"},
        ]
    )
    return chat_response.choices[0].message.content


In [20]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test.csv',header=[0])
df=df[['description','svg']]

### Sequentional execution (GPU time: 380s)

In [21]:
# import time
# from tqdm import tqdm

# tqdm.pandas()  # Enable tqdm for pandas apply

# start_time = time.time()

# df['response'] = df['description'].progress_apply(lambda x: get_reponse(x))

# end_time = time.time()
# print(f"Total time taken: {end_time - start_time:.2f} seconds")

100%|███████████████████████████████████████████| 76/76 [06:20<00:00,  5.01s/it]

Total time taken: 380.79 seconds


### Concurrent execution

In [33]:
import time
from tqdm import tqdm
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# Wrap tqdm over futures
def parallel_apply_with_tqdm(func, data, max_workers=12):
    results = [None] * len(data)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(func, data[i]): i for i in range(len(data))}
        for future in tqdm(as_completed(futures), total=len(data)):
            idx = futures[future]
            try:
                results[idx] = future.result()
            except Exception as e:
                results[idx] = None
                print(f"Error at index {idx}: {e}")
    return results

# Example usage
start_time = time.time()

df['response'] = parallel_apply_with_tqdm(get_reponse, df['description'].tolist(), max_workers=12)

end_time = time.time()
print(f"Total time taken: {end_time - start_time:.2f} seconds")


100%|███████████████████████████████████████████| 76/76 [00:39<00:00,  1.92it/s]

Total time taken: 39.72 seconds


In [34]:
df['response'].iloc[0]

'<svg viewBox="0 0 200 200" width="200" height="200" xmlns="http://www.w3.org/2000/svg">\n  <!-- Background -->\n  <rect x="0" y="0" width="200" height="200" fill="75deg(FFFA59)" />\n\n  <!-- Trees -->\n  <g transform="translate(20, 150)">\n    <polygon points="10,10 0,0 20,0" fill="FF8C00" />\n    <polygon points="15,15 5,5 15,5" fill="FF4500" />\n  </g>\n  <g transform="translate(60, 150)">\n    <polygon points="10,10 0,0 20,0" fill="FFD700" />\n    <polygon points="15,15 5,5 15,5" fill="FFA500" />\n  </g>\n  <g transform="translate(110, 150)">\n    <polygon points="10,10 0,0 20,0" fill="FF8C00" />\n    <polygon points="15,15 5,5 15,5" fill="FF4500" />\n  </g>\n  <g transform="translate(160, 150)">\n    <polygon points="10,10 0,0 20,0" fill="FFD700" />\n    <polygon points="15,15 5,5 15,5" fill="FFA500" />\n  </g>\n\n  <!-- Ground -->\n  <rect x="0" y="160" width="200" height="40" fill="FFB6C1" />\n\n  <!-- Leaves -->\n  <g fill="FF8C00" opacity="0.8">\n    <ellipse cx="10" cy="10" r